In [15]:
import joblib
import pandas as pd

In [16]:
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import classification_report

In [17]:
model = joblib.load("osmd_tuned_pipeline.pkl")
print("Model loaded successfully !")

Model loaded successfully !


In [18]:
calibrated_model = CalibratedClassifierCV(
    estimator = model,
    method = 'sigmoid',
    cv = 5
)

print("Calibrated Model created successfully !")

Calibrated Model created successfully !


In [19]:
import os
print(os.getcwd())

c:\Users\kk425\OneDrive\OSMD\src


In [20]:
df_clean = pd.read_csv("../data/processed/cleaned_messages.csv")

X = df_clean["message"]
y = df_clean["label"]

print("Dataset Shape:", df_clean.shape)
print("Labels:")
print(y.value_counts())

Dataset Shape: (5169, 3)
Labels:
label
ham     4516
spam     653
Name: count, dtype: int64


In [23]:
from sklearn.model_selection import train_test_split 

X_train, X_test, y_train, y_test = train_test_split(
                                                     X, y, test_size = 0.2, 
                                                     random_state = 42, stratify = y
                                                    )

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 4135
Testing samples: 1034


In [21]:
print(type(model))
print(model)

<class 'sklearn.pipeline.Pipeline'>
Pipeline(steps=[('tfidfvectorizer',
                 TfidfVectorizer(analyzer='char', min_df=2,
                                 ngram_range=(3, 5))),
                ('linearsvc', LinearSVC(C=10))])


In [24]:
y_pred = model.predict(X_test)

print("Predictions completed!")
print("Total predictions:", len(y_pred))

Predictions completed!
Total predictions: 1034


In [25]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Accuracy:", round(accuracy_score(y_test, y_pred) * 100, 2), "%")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred, labels = ["ham", "spam"]))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 98.84 %

Confusion Matrix:
[[902   1]
 [ 11 120]]

Classification Report:
              precision    recall  f1-score   support

         ham       0.99      1.00      0.99       903
        spam       0.99      0.92      0.95       131

    accuracy                           0.99      1034
   macro avg       0.99      0.96      0.97      1034
weighted avg       0.99      0.99      0.99      1034



In [26]:
calibrated_model.fit(X_train, y_train)
print("Calibrated Model training completed!")

Calibrated Model training completed!


In [27]:
message = ["Congratulations! You have won a free prize. Click here now!"]

probabilities = calibrated_model.predict_proba(message)

print("Classes:", calibrated_model.classes_)
print("Probabilities:", probabilities[0])

Classes: ['ham' 'spam']
Probabilities: [0.02351871 0.97648129]


In [28]:
spam_index = list(calibrated_model.classes_).index("spam")

spam_probability = probabilities[0][spam_index]
prediction = calibrated_model.predict(message)[0]

print("Prediction:", prediction.upper())
print("Spam Probability:", round(spam_probability * 100, 2), "%")

Prediction: SPAM
Spam Probability: 97.65 %


In [30]:
y_pred_calibrated = calibrated_model.predict(X_test)

print("Calibrated Model Evaluation")
print("Accuracy:", round(accuracy_score(y_test, y_pred_calibrated) * 100, 2), "%")
print(confusion_matrix(y_test, y_pred_calibrated, labels=["ham", "spam"]))
print(classification_report(y_test, y_pred_calibrated))

Calibrated Model Evaluation
Accuracy: 99.23 %
[[902   1]
 [  7 124]]
              precision    recall  f1-score   support

         ham       0.99      1.00      1.00       903
        spam       0.99      0.95      0.97       131

    accuracy                           0.99      1034
   macro avg       0.99      0.97      0.98      1034
weighted avg       0.99      0.99      0.99      1034



In [31]:
print("Old Model Accuracy: 98.84%")
print("Calibrated Model Accuracy: 99.23%")
print("Old spam missed: 11")
print("Calibrated spam missed: 7")

Old Model Accuracy: 98.84%
Calibrated Model Accuracy: 99.23%
Old spam missed: 11
Calibrated spam missed: 7


In [32]:
joblib.dump(calibrated_model, "osmd_calibrated_model.pkl")
print("Calibrated model saved!")

Calibrated model saved!
